# Step 1: Load Dataset

## Objective

In this step, the Superstore dataset is loaded into a PySpark DataFrame from the Databricks Volume. The dataset is inspected to understand its structure before performing any transformations.

## Why are we doing this?

The first step in any data engineering pipeline is to ingest the raw data into the processing engine. This allows us to inspect the dataset before applying cleaning and transformations.

In [0]:
from pyspark.sql.functions import col

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load("/Volumes/delta_lake_workspace/default/delta_assignment_data/Sample - Superstore.csv")
)

In [0]:
display(df.limit(50))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


In [0]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



# Step 2: Perform Basic Cleaning (Handle Nulls, Remove Duplicates)

## Objective

In this step, the dataset is cleaned by standardizing column names, checking for missing values, removing duplicate records (if any), and converting columns to the appropriate data types.

## Why are we doing this?

Data cleaning improves data quality and prepares the dataset for reliable processing. Clean data is essential before storing it in Delta Lake and performing incremental updates using the MERGE operation.

In [0]:
#Standardize Column Names
df = df.toDF(*[
    c.strip()
     .lower()
     .replace(" ", "_")
     .replace("-", "_")
    for c in df.columns
])

In [0]:
#Check Null Values
from pyspark.sql.functions import sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
#Check Duplicate Records
total_rows = df.count()
unique_rows = df.distinct().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", total_rows - unique_rows)

Total Rows: 9994
Unique Rows: 9994
Duplicate Rows: 0


In [0]:
#Convert Data Types
from pyspark.sql.functions import col

df = (
    df
    .withColumn("sales", col("sales").cast("double"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("discount", col("discount").cast("double"))
)

In [0]:
df.printSchema()

root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)



In [0]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/delta_lake_workspace/default/delta_assignment_data/Sample - Superstore.csv")
)

In [0]:
#Create the Delta Table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_lake_workspace.default.superstore_master")

# Step 3: Create a Second Dataset Simulating New Incremental Data

## Objective

In this step, we create a second dataset that simulates incremental data arriving from a source system. The incremental dataset contains both updated existing records and completely new records.

## Why are we doing this?

In real-world data engineering pipelines, new data arrives periodically. Some records update existing data, while others represent new entries. This incremental dataset will be used in the next step to demonstrate the Delta Lake MERGE operation.

In [0]:
#Read the Master Delta Table
df = spark.table("delta_lake_workspace.default.superstore_master")

In [0]:
display(df.limit(5))

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


In [0]:
#Create Updated Records
from pyspark.sql.functions import when, col

updated_df = (
    df
    .withColumn(
        "sales",
        when(col("row_id") == 1, 500.00)
        .when(col("row_id") == 2, 750.00)
        .when(col("row_id") == 3, 900.00)
        .when(col("row_id") == 4, 450.00)
        .when(col("row_id") == 5, 600.00)
        .otherwise(col("sales"))
    )
    .withColumn(
        "profit",
        when(col("row_id") == 1, 120.00)
        .when(col("row_id") == 2, 180.00)
        .when(col("row_id") == 3, 250.00)
        .when(col("row_id") == 4, 95.00)
        .when(col("row_id") == 5, 160.00)
        .otherwise(col("profit"))
    )
    .filter(col("row_id").isin([1, 2, 3, 4, 5]))
)

display(updated_df)

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,500.0,2,0.0,120.0
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",750.0,3,0.0,180.0
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,900.0,2,0.0,250.0
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,450.0,5,0.45,95.0
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,600.0,2,0.2,160.0


In [0]:
#Create New Records
from pyspark.sql import Row

new_records = [

    Row(
        row_id=10001,
        order_id="CA-2026-10001",
        order_date="2026-07-20",
        ship_date="2026-07-23",
        ship_mode="Second Class",
        customer_id="AJ-10001",
        customer_name="Akhil Jagtap",
        segment="Consumer",
        country="United States",
        city="Seattle",
        state="Washington",
        postal_code=98101,
        region="West",
        product_id="OFF-PA-100001",
        category="Office Supplies",
        sub_category="Paper",
        product_name="Premium Copy Paper",
        sales=550.25,
        quantity=4,
        discount=0.0,
        profit=180.50
    ),

    Row(
        row_id=10002,
        order_id="CA-2026-10002",
        order_date="2026-07-21",
        ship_date="2026-07-24",
        ship_mode="Standard Class",
        customer_id="RS-10002",
        customer_name="Rahul Sharma",
        segment="Corporate",
        country="United States",
        city="Chicago",
        state="Illinois",
        postal_code=60601,
        region="Central",
        product_id="TEC-PH-100002",
        category="Technology",
        sub_category="Phones",
        product_name="Wireless Phone",
        sales=899.99,
        quantity=2,
        discount=0.10,
        profit=250.75
    ),

    Row(
        row_id=10003,
        order_id="CA-2026-10003",
        order_date="2026-07-22",
        ship_date="2026-07-25",
        ship_mode="First Class",
        customer_id="PS-10003",
        customer_name="Priya Singh",
        segment="Home Office",
        country="United States",
        city="Dallas",
        state="Texas",
        postal_code=75201,
        region="Central",
        product_id="FUR-CH-100003",
        category="Furniture",
        sub_category="Chairs",
        product_name="Executive Office Chair",
        sales=725.40,
        quantity=1,
        discount=0.15,
        profit=210.35
    ),

    Row(
        row_id=10004,
        order_id="CA-2026-10004",
        order_date="2026-07-22",
        ship_date="2026-07-26",
        ship_mode="Standard Class",
        customer_id="AV-10004",
        customer_name="Aman Verma",
        segment="Consumer",
        country="United States",
        city="Los Angeles",
        state="California",
        postal_code=90001,
        region="West",
        product_id="TEC-AC-100004",
        category="Technology",
        sub_category="Accessories",
        product_name="Wireless Mouse",
        sales=149.99,
        quantity=3,
        discount=0.05,
        profit=55.25
    ),

    Row(
        row_id=10005,
        order_id="CA-2026-10005",
        order_date="2026-07-23",
        ship_date="2026-07-27",
        ship_mode="Second Class",
        customer_id="SP-10005",
        customer_name="Sneha Patel",
        segment="Corporate",
        country="United States",
        city="New York",
        state="New York",
        postal_code=10001,
        region="East",
        product_id="OFF-BI-100005",
        category="Office Supplies",
        sub_category="Binders",
        product_name="Premium Binder",
        sales=320.60,
        quantity=5,
        discount=0.0,
        profit=95.40
    )

]

In [0]:
#Create Incremental DataFrame
new_df = spark.createDataFrame(new_records)

incremental_df = updated_df.unionByName(new_df)

display(incremental_df)

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,500.0,2,0.0,120.0
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",750.0,3,0.0,180.0
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,900.0,2,0.0,250.0
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,450.0,5,0.45,95.0
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,600.0,2,0.2,160.0
10001,CA-2026-10001,2026-07-20,2026-07-23,Second Class,AJ-10001,Akhil Jagtap,Consumer,United States,Seattle,Washington,98101,West,OFF-PA-100001,Office Supplies,Paper,Premium Copy Paper,550.25,4,0.0,180.5
10002,CA-2026-10002,2026-07-21,2026-07-24,Standard Class,RS-10002,Rahul Sharma,Corporate,United States,Chicago,Illinois,60601,Central,TEC-PH-100002,Technology,Phones,Wireless Phone,899.99,2,0.1,250.75
10003,CA-2026-10003,2026-07-22,2026-07-25,First Class,PS-10003,Priya Singh,Home Office,United States,Dallas,Texas,75201,Central,FUR-CH-100003,Furniture,Chairs,Executive Office Chair,725.4,1,0.15,210.35
10004,CA-2026-10004,2026-07-22,2026-07-26,Standard Class,AV-10004,Aman Verma,Consumer,United States,Los Angeles,California,90001,West,TEC-AC-100004,Technology,Accessories,Wireless Mouse,149.99,3,0.05,55.25
10005,CA-2026-10005,2026-07-23,2026-07-27,Second Class,SP-10005,Sneha Patel,Corporate,United States,New York,New York,10001,East,OFF-BI-100005,Office Supplies,Binders,Premium Binder,320.6,5,0.0,95.4


# Step 4: Apply MERGE Operation

## Objective

In this step, the incremental dataset is merged into the master Delta table using the Delta Lake `MERGE` operation.

## Why are we doing this?

The MERGE operation enables upsert functionality by updating existing records and inserting new records in a single transaction. This is a common approach for handling incremental data in modern data engineering pipelines.

In [0]:
#Import DeltaTable
from delta.tables import DeltaTable

In [0]:
#Load the Master Delta Table
delta_table = DeltaTable.forName(
    spark,
    "delta_lake_workspace.default.superstore_master"
)

In [0]:
#Perform MERGE
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.row_id = source.row_id"
    )
    .whenMatchedUpdate(
        set={
            "sales": "source.sales",
            "profit": "source.profit"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#Verify the Updated Records
display(
    spark.table("delta_lake_workspace.default.superstore_master")
    .filter("row_id <= 5")
)

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,500.0,2,0.0,120.0
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",750.0,3,0.0,180.0
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,900.0,2,0.0,250.0
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,450.0,5,0.45,95.0
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,600.0,2,0.2,160.0


In [0]:
#Verify New Records
display(
    spark.table("delta_lake_workspace.default.superstore_master")
    .filter("row_id >= 10001")
)

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
10004,CA-2026-10004,2026-07-22,2026-07-26,Standard Class,AV-10004,Aman Verma,Consumer,United States,Los Angeles,California,90001,West,TEC-AC-100004,Technology,Accessories,Wireless Mouse,149.99,3,0.05,55.25
10001,CA-2026-10001,2026-07-20,2026-07-23,Second Class,AJ-10001,Akhil Jagtap,Consumer,United States,Seattle,Washington,98101,West,OFF-PA-100001,Office Supplies,Paper,Premium Copy Paper,550.25,4,0.0,180.5
10005,CA-2026-10005,2026-07-23,2026-07-27,Second Class,SP-10005,Sneha Patel,Corporate,United States,New York,New York,10001,East,OFF-BI-100005,Office Supplies,Binders,Premium Binder,320.6,5,0.0,95.4
10003,CA-2026-10003,2026-07-22,2026-07-25,First Class,PS-10003,Priya Singh,Home Office,United States,Dallas,Texas,75201,Central,FUR-CH-100003,Furniture,Chairs,Executive Office Chair,725.4,1,0.15,210.35
10002,CA-2026-10002,2026-07-21,2026-07-24,Standard Class,RS-10002,Rahul Sharma,Corporate,United States,Chicago,Illinois,60601,Central,TEC-PH-100002,Technology,Phones,Wireless Phone,899.99,2,0.1,250.75


# Step 5: Validate Results (Row Count, Duplicates)

## Objective

In this step, we validate the results of the MERGE operation by checking the total number of records, verifying that no duplicate records exist, and confirming that the updated and newly inserted records are present in the Delta table.

## Why are we doing this?

Validation ensures that the MERGE operation was successful and that the integrity of the dataset has been maintained after processing the incremental data.

In [0]:
#Read the Final Delta Table
final_df = spark.table("delta_lake_workspace.default.superstore_master")

In [0]:
#Validate Row Count
print("Total Rows:", final_df.count())

Total Rows: 9999


In [0]:
#Validate Duplicate Records
total_rows = final_df.count()
unique_rows = final_df.select("row_id").distinct().count()

print("Total Rows:", total_rows)
print("Unique Row IDs:", unique_rows)
print("Duplicate Row IDs:", total_rows - unique_rows)

Total Rows: 9999
Unique Row IDs: 9999
Duplicate Row IDs: 0


# Step 6: Display Final Dataset and Summary

## Objective

In this step, the final Delta table is displayed after the successful MERGE operation. A brief summary is provided to highlight the tasks completed during the assignment and the final outcome.

In [0]:
#Display Final Dataset
final_df = spark.table("delta_lake_workspace.default.superstore_master")

display(final_df.limit(20))

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47
11,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.184,9,0.2,85.3092
12,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002033,Technology,Phones,Konftel 250 Conference�phone�- Charcoal black,911.424,4,0.2,68.3568
13,CA-2017-114412,2017-04-15,2017-04-20,Standard Class,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552,3,0.2,5.4432
14,CA-2016-161389,2016-12-05,2016-12-10,Standard Class,IM-15070,Irene Maddox,Consumer,United States,Seattle,Washington,98103,West,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,407.976,3,0.2,132.5922
15,US-2015-118983,2015-11-22,2015-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,Texas,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,"Holmes Replacement Filter for HEPA Air Cleaner, Very Large Room, HEPA Filter",68.81,5,0.8,-123.858


In [0]:
#Summary
print("Delta Lake Assignment Summary")
print("-" * 40)
print("✔ Dataset loaded successfully")
print("✔ Basic data cleaning completed")
print("✔ Incremental dataset created")
print("✔ MERGE operation performed successfully")
print("✔ Existing records updated")
print("✔ New records inserted")
print("✔ Validation completed successfully")
print(f"✔ Final Row Count: {final_df.count()}")
print("✔ Duplicate Row IDs: 0")

Delta Lake Assignment Summary
----------------------------------------
✔ Dataset loaded successfully
✔ Basic data cleaning completed
✔ Incremental dataset created
✔ MERGE operation performed successfully
✔ Existing records updated
✔ New records inserted
✔ Validation completed successfully
✔ Final Row Count: 9999
✔ Duplicate Row IDs: 0
